# LeetCode 830 — Positions of Large Groups bằng Python

**Link đề:** https://leetcode.com/problems/positions-of-large-groups/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def largeGroupPositions(self, s: str) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **nhóm liên tiếp (run / group)** các ký tự giống nhau và cách mô tả nó bằng đoạn `[start, end]` (đóng hai đầu).
2. Học kỹ thuật **hai con trỏ "đầu nhóm – cuối nhóm"** (group by run) — quét một lượt `O(n)`.
3. Hiểu mẹo **xử lý nhóm cuối cùng** (điều kiện `i == n - 1 or s[i] != s[i + 1]`) — lỗi kinh điển của dạng bài này.
4. Biết các công cụ Python thay thế: `itertools.groupby`, biểu thức chính quy `re.finditer`.
5. Trace, chứng minh, test biên và mở rộng sang *run-length encoding*, *Consecutive Characters*.

## 1. Đề bài và ví dụ

Chuỗi `s` gồm chữ thường. Các ký tự giống nhau đứng **liền nhau** tạo thành một **nhóm**. Ví dụ `"abbxxxxzyy"` có các nhóm `"a"`, `"bb"`, `"xxxx"`, `"z"`, `"yy"`.

Mỗi nhóm được mô tả bởi đoạn `[start, end]` — chỉ số đầu và cuối, **tính cả hai đầu**. `"xxxx"` ở trên là `[3, 6]`.

Nhóm **lớn** nếu có **từ 3 ký tự trở lên**. Trả về các đoạn của mọi nhóm lớn, **sắp xếp tăng dần theo start**.

| s | Kết quả | Giải thích |
|---|---|---|
| `"abbxxxxzzy"` | `[[3,6]]` | chỉ `"xxxx"` lớn |
| `"abc"` | `[]` | không nhóm nào dài ≥ 3 |
| `"abcdddeeeeaabbbcd"` | `[[3,5],[6,9],[12,14]]` | `"ddd"`, `"eeee"`, `"bbb"` |

**Constraints:** `1 <= len(s) <= 1000`; `s` chỉ gồm chữ thường.

> Lưu ý: độ dài nhóm `[start, end]` là `end - start + 1` (vì tính cả hai đầu).

In [1]:
s = "abbxxxxzzy"
for i, ch in enumerate(s):
    print(i, ch)

0 a
1 b
2 b
3 x
4 x
5 x
6 x
7 z
8 z
9 y


## 2. Nền tảng Python cần dùng

### 2.1. Chỉ số, so sánh ký tự kề nhau `s[i] != s[i + 1]`

Ranh giới giữa hai nhóm là nơi `s[i] != s[i + 1]`. Cẩn thận: khi `i = n - 1` thì `s[i + 1]` **vượt chỉ số** → `IndexError`.

In [2]:
s = "aab"
for i in range(len(s) - 1):
    print(i, s[i], s[i + 1], "ranh giới" if s[i] != s[i + 1] else "")
try:
    s[len(s)]
except IndexError as e:
    print("Truy cập s[n]:", e)

0 a a 
1 a b ranh giới
Truy cập s[n]: string index out of range


### 2.2. Đánh giá ngắn mạch (short-circuit) của `or`

`A or B`: nếu `A` đúng thì Python **không tính** `B`. Nhờ vậy viết `i == n - 1 or s[i] != s[i + 1]` an toàn: khi `i == n - 1`, vế phải không bao giờ chạy.

In [3]:
s = "aab"
n = len(s)
i = n - 1
print(i == n - 1 or s[i] != s[i + 1])   # True, không lỗi
def loud():
    print("  (vế phải được tính)")
    return True
print(True or loud())
print(False or loud())

True
True
  (vế phải được tính)
True


### 2.3. List lồng nhau và `append([a, b])`

Kết quả là list các list 2 phần tử. `res.append([start, end])` thêm một đoạn mới.

In [4]:
res = []
res.append([3, 5])
res.append([6, 9])
print(res, len(res), res[1], res[1][0])

[[3, 5], [6, 9]] 2 [6, 9] 6


### 2.4. Độ dài đoạn đóng `[start, end]`

Đoạn `[3, 6]` gồm 3, 4, 5, 6 → **4** phần tử = `6 - 3 + 1`. Quên `+ 1` là lỗi "lệch một" (off-by-one) rất phổ biến.

In [5]:
start, end = 3, 6
print(list(range(start, end + 1)), "độ dài =", end - start + 1)

[3, 4, 5, 6] độ dài = 4


## 3. Cách 1 — Trực giác: với mỗi vị trí, đi xa nhất có thể

Ý tưởng tự nhiên: với mỗi `i`, đếm xem ký tự `s[i]` lặp liên tiếp tới đâu (`j`). Nếu `i` là **đầu nhóm** (tức `i == 0` hoặc `s[i-1] != s[i]`) và độ dài `j - i >= 3` thì ghi `[i, j - 1]`.

Cách này không nhảy qua nhóm nên với chuỗi toàn một ký tự `"aaaa...a"`, mỗi `i` lại đi tới cuối → `O(n²)`.

In [6]:
def large_groups_naive(s: str) -> list[list[int]]:
    n = len(s)
    res = []
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        is_group_start = (i == 0 or s[i - 1] != s[i])
        if is_group_start and j - i >= 3:
            res.append([i, j - 1])
    return res

print(large_groups_naive("abbxxxxzzy"))
print(large_groups_naive("abc"))
print(large_groups_naive("abcdddeeeeaabbbcd"))

[[3, 6]]
[]
[[3, 5], [6, 9], [12, 14]]


### Nút thắt

Đếm số bước của vòng `while` trên chuỗi toàn `'a'`: tổng là `n + (n-1) + ... + 1 = n(n+1)/2`.

In [7]:
def naive_steps(s):
    n = len(s); steps = 0
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            j += 1; steps += 1
    return steps

for n in [10, 100, 1000]:
    print(f"n={n:>4}, s='a'*n -> {naive_steps('a' * n):>7} bước")

n=  10, s='a'*n ->      55 bước
n= 100, s='a'*n ->    5050 bước
n=1000, s='a'*n ->  500500 bước


## 4. Insight — mỗi ký tự chỉ cần thuộc về **một** nhóm

Các nhóm **không chồng lên nhau** và phủ kín chuỗi. Vậy khi đã đi hết một nhóm `[start, end]`, ta **nhảy thẳng** tới `end + 1` để bắt đầu nhóm mới — không cần thử lại các vị trí bên trong.

Cách viết gọn nhất: duyệt `i` một lần, giữ `start` = chỉ số đầu nhóm hiện tại. Tại mỗi `i`, hỏi: **"`i` có phải ký tự cuối của nhóm không?"** — đúng khi `i == n - 1` (hết chuỗi) hoặc `s[i] != s[i + 1]` (ký tự sau khác). Nếu đúng:
1. Nhóm hiện tại là `[start, i]`, độ dài `i - start + 1`.
2. Nếu độ dài `>= 3` → ghi lại.
3. Nhóm mới bắt đầu ở `i + 1` → `start = i + 1`.

Vì ta quét trái → phải, các đoạn tự động **tăng dần theo start** — không cần sắp xếp.

In [8]:
s = "abbxxxxzzy"
n = len(s)
for i in range(n):
    is_end = (i == n - 1 or s[i] != s[i + 1])
    print(i, s[i], "<- cuối nhóm" if is_end else "")

0 a <- cuối nhóm
1 b 
2 b <- cuối nhóm
3 x 
4 x 
5 x 
6 x <- cuối nhóm
7 z 
8 z <- cuối nhóm
9 y <- cuối nhóm


## 5. Lời giải chính — hai con trỏ `start` / `i` (từng đoạn rồi ghép)

**Đoạn 1 — khởi tạo:** `res = []`, `start = 0`.

**Đoạn 2 — duyệt:** `for i in range(n)`.

**Đoạn 3 — phát hiện cuối nhóm:** `if i == n - 1 or s[i] != s[i + 1]:`

**Đoạn 4 — xử lý nhóm:** nếu `i - start + 1 >= 3` thì `res.append([start, i])`; sau đó `start = i + 1`.

In [9]:
def large_groups(s: str) -> list[list[int]]:
    n = len(s)
    res = []
    start = 0
    for i in range(n):
        if i == n - 1 or s[i] != s[i + 1]:      # i là ký tự cuối của nhóm
            if i - start + 1 >= 3:
                res.append([start, i])
            start = i + 1                        # nhóm tiếp theo bắt đầu ở đây
    return res

print(large_groups("abbxxxxzzy"))
print(large_groups("abc"))
print(large_groups("abcdddeeeeaabbbcd"))

[[3, 6]]
[]
[[3, 5], [6, 9], [12, 14]]


## 6. TRACE từng bước

### 6.1. Bảng trace cho `s = "aaabccc"` (n = 7) — có nhóm lớn **ở cuối chuỗi**

| i | s[i] | s[i+1] | cuối nhóm? | nhóm | độ dài | hành động | start sau |
|---|---|---|---|---|---|---|---|
| 0 | a | a | không | | | | 0 |
| 1 | a | a | không | | | | 0 |
| 2 | a | b | **có** | [0,2] | 3 | ghi [0,2] | 3 |
| 3 | b | c | **có** | [3,3] | 1 | bỏ qua | 4 |
| 4 | c | c | không | | | | 4 |
| 5 | c | c | không | | | | 4 |
| 6 | c | — | **có** (i = n−1) | [4,6] | 3 | ghi [4,6] | 7 |

Kết quả `[[0,2],[4,6]]`. Dòng cuối cho thấy vì sao cần `i == n - 1`: nếu thiếu, nhóm `"ccc"` bị bỏ sót.

In [10]:
def trace_large_groups(s: str) -> list[list[int]]:
    n = len(s)
    res = []
    start = 0
    print(f"s={s!r}, n={n}")
    for i in range(n):
        nxt = s[i + 1] if i + 1 < n else "—"
        is_end = i == n - 1 or s[i] != s[i + 1]
        msg = ""
        if is_end:
            length = i - start + 1
            if length >= 3:
                res.append([start, i])
                msg = f"nhóm [{start},{i}] dài {length} -> GHI"
            else:
                msg = f"nhóm [{start},{i}] dài {length} -> bỏ qua"
            start = i + 1
        print(f"i={i} s[i]={s[i]} s[i+1]={nxt} cuối_nhóm={is_end!s:5} {msg:30} start={start} res={res}")
    return res

trace_large_groups("aaabccc");

s='aaabccc', n=7
i=0 s[i]=a s[i+1]=a cuối_nhóm=False                                start=0 res=[]
i=1 s[i]=a s[i+1]=a cuối_nhóm=False                                start=0 res=[]
i=2 s[i]=a s[i+1]=b cuối_nhóm=True  nhóm [0,2] dài 3 -> GHI        start=3 res=[[0, 2]]
i=3 s[i]=b s[i+1]=c cuối_nhóm=True  nhóm [3,3] dài 1 -> bỏ qua     start=4 res=[[0, 2]]
i=4 s[i]=c s[i+1]=c cuối_nhóm=False                                start=4 res=[[0, 2]]
i=5 s[i]=c s[i+1]=c cuối_nhóm=False                                start=4 res=[[0, 2]]
i=6 s[i]=c s[i+1]=— cuối_nhóm=True  nhóm [4,6] dài 3 -> GHI        start=7 res=[[0, 2], [4, 6]]


### 6.2. Trace ví dụ đề 3

In [11]:
trace_large_groups("abcdddeeeeaabbbcd");

s='abcdddeeeeaabbbcd', n=17
i=0 s[i]=a s[i+1]=b cuối_nhóm=True  nhóm [0,0] dài 1 -> bỏ qua     start=1 res=[]
i=1 s[i]=b s[i+1]=c cuối_nhóm=True  nhóm [1,1] dài 1 -> bỏ qua     start=2 res=[]
i=2 s[i]=c s[i+1]=d cuối_nhóm=True  nhóm [2,2] dài 1 -> bỏ qua     start=3 res=[]
i=3 s[i]=d s[i+1]=d cuối_nhóm=False                                start=3 res=[]
i=4 s[i]=d s[i+1]=d cuối_nhóm=False                                start=3 res=[]
i=5 s[i]=d s[i+1]=e cuối_nhóm=True  nhóm [3,5] dài 3 -> GHI        start=6 res=[[3, 5]]
i=6 s[i]=e s[i+1]=e cuối_nhóm=False                                start=6 res=[[3, 5]]
i=7 s[i]=e s[i+1]=e cuối_nhóm=False                                start=6 res=[[3, 5]]
i=8 s[i]=e s[i+1]=e cuối_nhóm=False                                start=6 res=[[3, 5]]
i=9 s[i]=e s[i+1]=a cuối_nhóm=True  nhóm [6,9] dài 4 -> GHI        start=10 res=[[3, 5], [6, 9]]
i=10 s[i]=a s[i+1]=a cuối_nhóm=False                                start=10 res=[[3, 5], [6, 9]]
i=11 s[i]=a s[i

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def largeGroupPositions(self, s: str) -> list[list[int]]:
        n = len(s)
        res = []
        start = 0
        for i in range(n):
            if i == n - 1 or s[i] != s[i + 1]:
                if i - start + 1 >= 3:
                    res.append([start, i])
                start = i + 1
        return res

sol = Solution()
assert sol.largeGroupPositions("abbxxxxzzy") == [[3, 6]]
assert sol.largeGroupPositions("abc") == []
assert sol.largeGroupPositions("abcdddeeeeaabbbcd") == [[3, 5], [6, 9], [12, 14]]
print("Solution PASS 3 ví dụ đề")

Solution PASS 3 ví dụ đề


## 8. Vì sao đúng? Độ phức tạp

**Invariant:** ở đầu mỗi vòng lặp với chỉ số `i`, `start` là chỉ số đầu của nhóm chứa `i`, và `res` chứa đúng các nhóm lớn đã **kết thúc** trước `i`, theo thứ tự tăng dần.

- Ban đầu `i = 0`, `start = 0`, `res = []` — đúng.
- Nếu `i` không phải cuối nhóm thì `s[i + 1] == s[i]`, nên `i + 1` cùng nhóm, `start` giữ nguyên — đúng.
- Nếu `i` là cuối nhóm: nhóm `[start, i]` vừa kết thúc; ta ghi nếu đủ lớn, rồi `start = i + 1` là đầu nhóm kế tiếp — đúng.
- Kết thúc: `i = n - 1` luôn được coi là cuối nhóm, nên nhóm cuối cùng **không bị bỏ sót**.

**Độ phức tạp:** `n = len(s)`.
- Time `O(n)`: mỗi `i` làm việc hằng số.
- Space `O(1)` ngoài output (output tối đa `n/3` đoạn).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Naive (không nhảy) | `O(n²)` tệ nhất | `O(1)` |
| 2. Hai con trỏ start / i | `O(n)` | `O(1)` |
| 3. `while` lồng nhưng nhảy cả nhóm | `O(n)` | `O(1)` |
| 4. `itertools.groupby` | `O(n)` | `O(1)` (+ list nhỏ mỗi nhóm nếu `list(g)`) |
| 5. Regex | `O(n)` | `O(1)` |

## 9. Cách 3 — `while` lồng nhau nhưng **nhảy** cả nhóm

Giống cách 1 nhưng sau khi tìm được `j` (vị trí đầu tiên khác ký tự), gán `i = j`. Dù có hai vòng lặp lồng nhau, mỗi chỉ số chỉ được `j` đi qua **một lần** → tổng `O(n)`. Đây là mẫu "two pointers cùng chiều" rất hay dùng.

In [13]:
def large_groups_jump(s: str) -> list[list[int]]:
    n = len(s)
    res = []
    i = 0
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        # nhóm là [i, j - 1]
        if j - i >= 3:
            res.append([i, j - 1])
        i = j                       # nhảy sang nhóm kế
    return res

print(large_groups_jump("abcdddeeeeaabbbcd"))

[[3, 5], [6, 9], [12, 14]]


## 10. Cách 4 — `itertools.groupby`

`itertools.groupby(s)` gom các phần tử **liên tiếp** bằng nhau, cho ra từng cặp `(key, iterator_của_nhóm)`. Ta chỉ cần độ dài nhóm và giữ một biến `pos` để biết vị trí.

In [14]:
from itertools import groupby
for key, grp in groupby("abbxxxxzzy"):
    items = list(grp)
    print(key, items, len(items))

a ['a'] 1
b ['b', 'b'] 2
x ['x', 'x', 'x', 'x'] 4
z ['z', 'z'] 2
y ['y'] 1


In [15]:
from itertools import groupby

def large_groups_groupby(s: str) -> list[list[int]]:
    res = []
    pos = 0
    for _, grp in groupby(s):
        length = sum(1 for _ in grp)      # đếm phần tử của nhóm
        if length >= 3:
            res.append([pos, pos + length - 1])
        pos += length
    return res

print(large_groups_groupby("abcdddeeeeaabbbcd"))

[[3, 5], [6, 9], [12, 14]]


## 11. Cách 5 — Biểu thức chính quy (regex)

Mẫu `r"(.)\1{2,}"` nghĩa là: một ký tự bất kỳ `(.)` (được "bắt" làm nhóm 1), rồi chính ký tự đó `\1` lặp **ít nhất 2 lần nữa** `{2,}` → tổng ≥ 3. `re.finditer` trả về các kết quả khớp, mỗi cái có `.start()` và `.end()` (end là **sau** ký tự cuối, nên phải `- 1`).

In [16]:
import re
for m in re.finditer(r"(.)\1{2,}", "abcdddeeeeaabbbcd"):
    print(repr(m.group()), m.start(), m.end())

'ddd' 3 6
'eeee' 6 10
'bbb' 12 15


In [17]:
import re

class SolutionRegex:
    def largeGroupPositions(self, s: str) -> list[list[int]]:
        return [[m.start(), m.end() - 1] for m in re.finditer(r"(.)\1{2,}", s)]

print(SolutionRegex().largeGroupPositions("abcdddeeeeaabbbcd"))

[[3, 5], [6, 9], [12, 14]]


> Regex rất gọn nhưng là "hộp đen" — trong phỏng vấn nên trình bày cách hai con trỏ trước, regex chỉ để nói thêm.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Chuỗi rất ngắn: `"a"`, `"aa"`, `"aaa"`.
3. Nhóm lớn ở **đầu**, ở **cuối**, chiếm **toàn chuỗi**.
4. Hai nhóm lớn **liền nhau** khác ký tự: `"aaabbb"`.
5. Nhóm dài đúng 2 (không lớn) xen kẽ.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.largeGroupPositions("abbxxxxzzy"), [[3, 6]])
check("đề 2", sol.largeGroupPositions("abc"), [])
check("đề 3", sol.largeGroupPositions("abcdddeeeeaabbbcd"), [[3, 5], [6, 9], [12, 14]])
check("1 ký tự", sol.largeGroupPositions("a"), [])
check("dài 2", sol.largeGroupPositions("aa"), [])
check("dài 3 toàn chuỗi", sol.largeGroupPositions("aaa"), [[0, 2]])
check("lớn ở cuối", sol.largeGroupPositions("abbb"), [[1, 3]])
check("lớn ở đầu", sol.largeGroupPositions("aaab"), [[0, 2]])
check("liền nhau", sol.largeGroupPositions("aaabbb"), [[0, 2], [3, 5]])
check("toàn 2", sol.largeGroupPositions("aabbcc"), [])

PASS đề 1: [[3, 6]]
PASS đề 2: []
PASS đề 3: [[3, 5], [6, 9], [12, 14]]
PASS 1 ký tự: []
PASS dài 2: []
PASS dài 3 toàn chuỗi: [[0, 2]]
PASS lớn ở cuối: [[1, 3]]
PASS lớn ở đầu: [[0, 2]]
PASS liền nhau: [[0, 2], [3, 5]]
PASS toàn 2: []


### 12.1. Đối chiếu 6 cài đặt (có seed)

Dùng bảng chữ nhỏ `"ab"` để dễ sinh các nhóm dài.

In [19]:
import random
random.seed(830)
impls = [large_groups_naive, large_groups, large_groups_jump, large_groups_groupby,
         Solution().largeGroupPositions, SolutionRegex().largeGroupPositions]
for _ in range(500):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 20)))
    expected = large_groups_naive(s)
    for f in impls:
        assert f(s) == expected, (s, f)
    # tính chất: mỗi đoạn dài >= 3, ký tự đồng nhất, và tăng dần
    for st, en in expected:
        assert en - st + 1 >= 3 and len(set(s[st:en + 1])) == 1
    assert expected == sorted(expected)
print("500 chuỗi ngẫu nhiên: 6 cài đặt khớp nhau, tính chất đúng")

500 chuỗi ngẫu nhiên: 6 cài đặt khớp nhau, tính chất đúng


## 13. Lỗi người mới hay gặp

### Lỗi 1: bỏ sót nhóm cuối chuỗi

Chỉ chốt nhóm khi `s[i] != s[i + 1]` và duyệt `range(n - 1)` → nhóm cuối không bao giờ được chốt.

In [20]:
def wrong_last_group(s):
    res = []; start = 0
    for i in range(len(s) - 1):
        if s[i] != s[i + 1]:
            if i - start + 1 >= 3:
                res.append([start, i])
            start = i + 1
    return res

print("sai :", wrong_last_group("abccc"))
print("đúng:", Solution().largeGroupPositions("abccc"))

sai : []
đúng: [[2, 4]]


### Lỗi 2: off-by-one khi tính độ dài

`i - start >= 3` (quên `+ 1`) → nhóm dài đúng 3 bị bỏ qua.

In [21]:
def wrong_length(s):
    n = len(s); res = []; start = 0
    for i in range(n):
        if i == n - 1 or s[i] != s[i + 1]:
            if i - start >= 3:                 # SAI
                res.append([start, i])
            start = i + 1
    return res

print("sai :", wrong_length("aaab"))
print("đúng:", Solution().largeGroupPositions("aaab"))

sai : []
đúng: [[0, 2]]


### Lỗi 3: đảo thứ tự điều kiện `or`

`s[i] != s[i + 1] or i == n - 1` → tại `i = n - 1`, vế trái chạy trước và gây `IndexError`.

In [22]:
s = "aab"; n = len(s); i = n - 1
try:
    print(s[i] != s[i + 1] or i == n - 1)
except IndexError as e:
    print("IndexError:", e)
print(i == n - 1 or s[i] != s[i + 1], "<- đúng thứ tự")

IndexError: string index out of range
True <- đúng thứ tự


### Lỗi 4: dùng regex `end()` mà quên `- 1`

`m.end()` là vị trí **sau** ký tự cuối (giống `stop` của slicing) → đoạn bị dài thêm 1.

In [23]:
import re
m = re.search(r"(.)\1{2,}", "abbb")
print("sai :", [m.start(), m.end()], "| đúng:", [m.start(), m.end() - 1])

sai : [1, 4] | đúng: [1, 3]


## 14. Mẹo nhận dạng pattern

- Đề nói "**liên tiếp** các ký tự/phần tử giống nhau", "nhóm", "run", "chuỗi con đồng nhất dài nhất" → **group by run**: giữ `start`, chốt nhóm khi phần tử kế tiếp khác hoặc hết mảng.
- Hai mẫu viết hay dùng:
  - `for i in range(n): if i == n-1 or a[i] != a[i+1]: ...chốt [start, i]...; start = i + 1`
  - `i = 0; while i < n: j = i; while j < n and a[j] == a[i]: j += 1; ...[i, j-1]...; i = j`
- Bài cùng họ: *Consecutive Characters (1446)*, *String Compression (443)*, *Count and Say (38)*, *Max Consecutive Ones (485)*, *Count Binary Substrings (696)*.

## 15. Biến thể

### 15.1. Run-length encoding: `"aaabcc" → "a3b1c2"`

In [24]:
def rle(s: str) -> str:
    out = []
    i, n = 0, len(s)
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        out.append(s[i] + str(j - i))
        i = j
    return "".join(out)

print(rle("aaabcc"), rle("a"), rle("abcdddeeee"))

a3b1c2 a1 a1b1c1d3e4


### 15.2. Ngưỡng "lớn" tùy biến `k` và trả kèm ký tự

In [25]:
def groups_at_least(s: str, k: int) -> list[tuple]:
    n = len(s); res = []; start = 0
    for i in range(n):
        if i == n - 1 or s[i] != s[i + 1]:
            if i - start + 1 >= k:
                res.append((s[i], start, i))
            start = i + 1
    return res

print(groups_at_least("abcdddeeeeaabbbcd", 2))
print(groups_at_least("abcdddeeeeaabbbcd", 4))

[('d', 3, 5), ('e', 6, 9), ('a', 10, 11), ('b', 12, 14)]
[('e', 6, 9)]


### 15.3. Áp dụng cho list số nguyên

Cùng một khung với mảng số nguyên: tìm các đoạn có ≥ 3 số bằng nhau liên tiếp.

In [26]:
def large_runs_nums(a: list[int]) -> list[list[int]]:
    n = len(a); res = []; start = 0
    for i in range(n):
        if i == n - 1 or a[i] != a[i + 1]:
            if i - start + 1 >= 3:
                res.append([start, i])
            start = i + 1
    return res

print(large_runs_nums([7, 7, 7, 1, 2, 2, 2, 2, 5]))

[[0, 2], [4, 7]]


## 16. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Liệt kê mọi nhóm (kể cả nhóm nhỏ) của `"zzzyxxxxw"` dưới dạng `[start, end]` và cho biết kết quả của bài 830.

### Bài 2 — Dự đoán output
Không chạy, đoán: `[len(list(g)) for _, g in groupby("aabccc")]` và `True or 1/0`.

### Bài 3 — Điền TODO cho lời giải chính

In [27]:
def ex3_large_groups(s: str) -> list[list[int]]:
    n = len(s)
    res = []
    start = 0
    for i in range(n):
        # TODO: nếu i là cuối nhóm -> kiểm tra độ dài, ghi đoạn, cập nhật start
        pass
    return res

# assert ex3_large_groups("abcdddeeeeaabbbcd") == [[3, 5], [6, 9], [12, 14]]

### Bài 4 — Consecutive Characters (LeetCode 1446)
Trả về độ dài của nhóm **dài nhất**.

In [28]:
def ex4_max_power(s: str) -> int:
    # TODO
    pass

# assert ex4_max_power("leetcode") == 2
# assert ex4_max_power("abbcccddddeeeeedcba") == 5

### Bài 5 — Run-length decoding
Cho chuỗi dạng `"a3b1c2"` (mỗi ký tự theo sau là **một chữ số** 1–9), giải mã thành `"aaabcc"`.

In [29]:
def ex5_rle_decode(code: str) -> str:
    # TODO
    pass

# assert ex5_rle_decode("a3b1c2") == "aaabcc"

### Bài 6 — Mini challenge: Count Binary Substrings (LeetCode 696)
Cho chuỗi nhị phân `s`, đếm số chuỗi con có số `0` bằng số `1` và tất cả `0` liền nhau, tất cả `1` liền nhau (ví dụ `"0011"`, `"10"`). Gợi ý: tính độ dài các nhóm liên tiếp.

In [30]:
def ex6_count_binary_substrings(s: str) -> int:
    # TODO
    pass

# assert ex6_count_binary_substrings("00110011") == 6
# assert ex6_count_binary_substrings("10101") == 4

## 17. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 4: duyệt các nhóm, lấy max độ dài.
- Bài 5: đọc từng cặp (ký tự, chữ số).
- Bài 6: hai nhóm **kề nhau** có độ dài `x` và `y` đóng góp `min(x, y)` chuỗi con hợp lệ.

**Mức 2 — cú pháp**
- Bài 3: `if i == n - 1 or s[i] != s[i + 1]:` rồi `if i - start + 1 >= 3: res.append([start, i])`, `start = i + 1`.
- Bài 5: `for k in range(0, len(code), 2): ch, cnt = code[k], int(code[k + 1])`.

**Mức 3 — gần code**
- Bài 6: `lens = [độ dài các nhóm]`; `return sum(min(lens[k], lens[k + 1]) for k in range(len(lens) - 1))`.

## 18. Đáp án tham khảo và tổng kết

**Bài 1:** nhóm `[0,2]` (zzz), `[3,3]` (y), `[4,7]` (xxxx), `[8,8]` (w) → kết quả `[[0,2],[4,7]]`.

**Bài 2:** `[2, 1, 3]` và `True` (vế `1/0` không được tính nhờ short-circuit).

In [31]:
from itertools import groupby
print([len(list(g)) for _, g in groupby("aabccc")], True or 1/0)
assert Solution().largeGroupPositions("zzzyxxxxw") == [[0, 2], [4, 7]]

def ans3_large_groups(s: str) -> list[list[int]]:
    n = len(s); res = []; start = 0
    for i in range(n):
        if i == n - 1 or s[i] != s[i + 1]:
            if i - start + 1 >= 3:
                res.append([start, i])
            start = i + 1
    return res

def ans4_max_power(s: str) -> int:
    best = 0; start = 0; n = len(s)
    for i in range(n):
        if i == n - 1 or s[i] != s[i + 1]:
            best = max(best, i - start + 1)
            start = i + 1
    return best

def ans5_rle_decode(code: str) -> str:
    return "".join(code[k] * int(code[k + 1]) for k in range(0, len(code), 2))

def ans6_count_binary_substrings(s: str) -> int:
    lens = [len(list(g)) for _, g in groupby(s)]
    return sum(min(lens[k], lens[k + 1]) for k in range(len(lens) - 1))

assert ans3_large_groups("abcdddeeeeaabbbcd") == [[3, 5], [6, 9], [12, 14]]
assert ans4_max_power("leetcode") == 2
assert ans4_max_power("abbcccddddeeeeedcba") == 5
assert ans5_rle_decode("a3b1c2") == "aaabcc"
assert ans5_rle_decode(rle("abcdddeeee")) == "abcdddeeee"
assert ans6_count_binary_substrings("00110011") == 6
assert ans6_count_binary_substrings("10101") == 4
print("Đáp án bài 1-6 PASS")

[2, 1, 3] True
Đáp án bài 1-6 PASS


### Template nhớ nhanh — group by run

```python
res, start, n = [], 0, len(s)
for i in range(n):
    if i == n - 1 or s[i] != s[i + 1]:   # i là cuối nhóm (thứ tự or quan trọng!)
        length = i - start + 1             # đoạn đóng: +1
        if length >= 3:
            res.append([start, i])
        start = i + 1
return res
```

### Checklist trước khi submit
- [ ] Nhóm **cuối chuỗi** có được chốt không (`i == n - 1`)?
- [ ] Viết `i == n - 1` **trước** `s[i] != s[i + 1]` để không vượt chỉ số?
- [ ] Độ dài là `i - start + 1`, so sánh `>= 3`?
- [ ] Đoạn trả về là `[start, end]` **đóng** (end là chỉ số ký tự cuối)?
- [ ] Kết quả tăng dần theo start (tự động khi quét trái → phải)?
- [ ] Đã test `"aaa"`, `"aa"`, `"abbb"`, `"aaabbb"`?